# NumiNaMath Dataset Evaluation Notebook

This notebook demonstrates how to:
1. Load the NumiNaMath dataset from local files or HuggingFace.
2. Preprocess samples to extract the problem and boxed solution.
3. Call a frontier model API (e.g., GPT) using an API key from environment variables.
4. Run inference on 5 random test samples.
5. Compute cross-entropy between ground truth and prediction.


## 1. Import Required Libraries

We will use `datasets` for data loading, `re` for regex extraction, `os` for environment variables, and `random` for sampling.

In [1]:
import os
import re
import random
from datasets import load_dataset, Dataset, load_from_disk
from pathlib import Path
import requests
import numpy as np

In [2]:
#------------ CACHE ENVIRONMENT ----------------------------------------
# Set cache environment variables BEFORE importing datasets
PROJECT_ROOT = Path.cwd().parent  # Go up one level
hf_cache = PROJECT_ROOT / ".cache" / "huggingface"
hf_cache.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(hf_cache)
os.environ["HF_DATASETS_CACHE"] = str(hf_cache / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache / "transformers")

## 2. Load the NumiNaMath Dataset

Try to load the dataset from the local directory. If not found, download from HuggingFace.

In [3]:
# Define local dataset path
local_dataset_path = Path('../datasets/test/data-00000-of-00001.arrow').resolve()

# dataset ID
HF_DATASET_ID = "AI-MO/NuminaMath-TIR"
PROJECT_ROOT = Path.cwd().parent  # Go up one level
DATASET_DIR = PROJECT_ROOT / "datasets"

# logic
if local_dataset_path.exists():
    print(f"Loading dataset from local path: {local_dataset_path}")
    # dataset = Dataset.load_from_disk(str(local_dataset_path.parent.parent))
    dataset = load_from_disk(str(DATASET_DIR))
    test_split = dataset['test']
else:
    print("Local dataset not found. Downloading from HuggingFace...")
    dataset = load_dataset(HF_DATASET_ID)
    test_split = dataset['test']

print(f"Loaded {len(test_split)} test samples.")

Loading dataset from local path: /home/benjamin.deporte/GenAI_Math/datasets/test/data-00000-of-00001.arrow
Loaded 99 test samples.


## 3. Preprocessing: Extract Problem and Boxed Solution

Define a function to extract the problem and the boxed solution (using `\\boxed{...}` regex) from each sample. Handle missing boxed solutions gracefully.

In [50]:
def extract_problem_and_boxed_solution(sample):
    problem = sample.get('problem', None)
    solution = sample.get('solution', None)
    if problem is None or solution is None:
        raise ValueError("Sample missing 'problem' or 'solution' field.")
    # Match everything between \\boxed{ and the first closing }
    match = re.search(r"\\boxed\{(.*?)\}", solution, re.DOTALL)
    if not match:
        raise ValueError("Boxed solution not found in solution field.")
    boxed_solution = match.group(1)
    return problem, solution, boxed_solution

## 4. Set Up Model API Access

Read the API key from environment variables and define a function to call the model API (e.g., OpenAI GPT).

In [58]:
# Example for OpenAI GPT API (adjust endpoint/model as needed)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
assert OPENAI_API_KEY, "OPENAI_API_KEY not found in environment variables."

API_URL = "https://api.openai.com/v1/chat/completions"
MODEL_NAME = "gpt-3.5-turbo"  # Change as needed

def call_gpt_api_with_logprobs(problem, max_tokens=2048):
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    prompt = (
        "Solve the following math problem. "
        "Enclose your reasoning in <think> and </think>, and your final answer in <answer>...</answer> in LaTeX format. For example use \text{ } to enclose text answers."
        "Return only these tags.\n\n"
        f"Problem: {problem}"
    )
    data = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        # "logprobs": True,  # Request logprobs for each token
        # "top_logprobs": 5  # Number of top logprobs to return per token
    }
    response = requests.post(API_URL, headers=headers, json=data)
    response.raise_for_status()
    result = response.json()['choices'][0]['message']
    # logprobs = response.json()['choices'][0].get('logprobs', None)
    return result['content'].strip()    #, logprobs

## 5. Run Inference on some Random Test Samples

Randomly select 5 samples, extract the problem and boxed solution, call the model, and collect predictions.

In [59]:
results = []
sampled_indices = random.sample(range(len(test_split)), 3)

for idx in sampled_indices:
    sample = test_split[idx]
    try:
        problem, solution, boxed_solution = extract_problem_and_boxed_solution(sample)
    except ValueError as e:
        print(f"Skipping sample {idx}: {e}")
        continue
    try:
        prediction = call_gpt_api_with_logprobs(problem)
    except Exception as e:
        print(f"API call failed for sample {idx}: {e}")
        prediction = None, None
    results.append({
        'problem': problem,
        'solution': solution,
        'ground_truth': boxed_solution,
        'prediction': prediction,
        # 'logprobs': logprobs
    })

for i, res in enumerate(results):
    print(f"Sample {i+1}:")
    print(f"Problem: {res['problem']}")
    print(f"Raw solution: {res['solution'][-100:]}")
    print(f"Ground Truth: {res['ground_truth']}")
    print(f"Prediction: {res['prediction']}")
    print('-'*40)

Sample 1:
Problem: Antoine owns a strawberry farm that supplies strawberries to his local bakeries. The first bakery needs 2 sacks, the second bakery needs 4 sacks, and the third bakery needs 12 sacks of strawberries per week. How many sacks of strawberries does he need to supply all the bakeries in 4 weeks?
Raw solution: needs to supply \(\boxed{72}\) sacks of strawberries to all the bakeries over the course of 4 weeks.
Ground Truth: 72
Prediction: <think>
To find out how many sacks of strawberries Antoine needs to supply all the bakeries in 4 weeks, we first need to calculate the total number of sacks needed by all three bakeries in one week. Then we can multiply this total by 4 to account for 4 weeks.
The first bakery needs 2 sacks per week, the second bakery needs 4 sacks per week, and the third bakery needs 12 sacks per week. Adding these up gives us the total sacks needed per week.
</think>
<answer>
Total sacks needed per week = 2 + 4 + 12 = 18 sacks
Total sacks needed in 4 weeks

# Compute cross-entropy for each result

In [53]:
# --- Tokenizer management utility ---
import os
from pathlib import Path
from transformers import AutoTokenizer

# Ensure tokenizer directory is in project root/models/tokenizers
PROJECT_ROOT = Path.cwd().parent  # Go up one level from /notebooks
TOKENIZER_DIR = PROJECT_ROOT / "models" / "tokenizers"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

def get_tokenizer(tokenizer_name):
    local_tokenizer_path = TOKENIZER_DIR / tokenizer_name.replace("/", "_")
    if local_tokenizer_path.exists():
        tokenizer = AutoTokenizer.from_pretrained(
            str(local_tokenizer_path),
            cache_dir=str(TOKENIZER_DIR)
        )
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name,
            cache_dir=str(TOKENIZER_DIR)
        )
        tokenizer.save_pretrained(str(local_tokenizer_path))
    return tokenizer

In [54]:
# --- Updated cross-entropy computation for logprobs['content'] structure ---•
def compute_cross_entropy_from_logprobs_content(gt_answer, logprobs_content, tokenizer):
    """
    Compute cross-entropy between ground truth answer and model logprobs.
    - gt_answer: string (ground truth answer)
    - logprobs_content: list of dicts with 'token' and 'logprob'
    - tokenizer: HuggingFace tokenizer
    Returns: cross-entropy (float) or None if not alignable
    """
    # Tokenize ground truth answer
    gt_tokens = tokenizer.convert_ids_to_tokens(tokenizer(gt_answer, add_special_tokens=False)["input_ids"])
    pred_tokens = [t['token'] for t in logprobs_content]

    # Try to find the subsequence match
    def find_subsequence(sub, seq):
        for i in range(len(seq) - len(sub) + 1):
            if seq[i:i+len(sub)] == sub:
                return i
        return -1

    start_idx = find_subsequence(gt_tokens, pred_tokens)
    if start_idx == -1:
        # Try to join tokens and match as string (for BPE/wordpiece mismatch)
        gt_str = tokenizer.convert_tokens_to_string(gt_tokens).strip()
        pred_str = tokenizer.convert_tokens_to_string(pred_tokens).strip()
        if gt_str in pred_str:
            # Try to align by string offset
            idx = pred_str.index(gt_str)
            # Find which token this offset corresponds to
            running = ''
            for i, t in enumerate(pred_tokens):
                running += tokenizer.convert_tokens_to_string([t])
                if running.strip().endswith(gt_str):
                    start_idx = i - len(gt_tokens) + 1
                    break
        else:
            return None
    if start_idx < 0 or start_idx + len(gt_tokens) > len(pred_tokens):
        return None
    # Compute cross-entropy
    logprobs = [logprobs_content[start_idx + i]['logprob'] for i in range(len(gt_tokens))]
    xent = -sum(logprobs) / len(logprobs)
    return xent

# Example usage for sample 1:
# logprobs_content = results[0]['logprobs']['content']
# gt = gt_answers[0]
# ce = compute_cross_entropy_from_logprobs_content(gt, logprobs_content, tokenizer)
# print(f"Sample 1 Cross-Entropy (answer only): {ce if ce is not None else 'N/A'}")


In [55]:
import re

def extract_answer_from_prediction(prediction):
    # Extract content between <answer> and </answer>
    match = re.search(r"<answer>(.*?)</answer>", prediction, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

def compute_cross_entropy_from_logprobs(gt, pred, logprobs, tokenizer_name="gpt2"):
    tokenizer = get_tokenizer(tokenizer_name)
    gt_tokens = tokenizer(gt, add_special_tokens=False)["input_ids"]
    if logprobs is None or 'tokens' not in logprobs or 'token_logprobs' not in logprobs:
        return None
    pred_token_strs = logprobs['tokens']
    pred_token_ids = tokenizer.convert_tokens_to_ids(pred_token_strs)
    pred_logprobs = logprobs['token_logprobs']
    # Search for gt_tokens as a subsequence in pred_token_ids
    for start in range(len(pred_token_ids) - len(gt_tokens) + 1):
        if pred_token_ids[start:start+len(gt_tokens)] == gt_tokens:
            ce = -sum(pred_logprobs[start:start+len(gt_tokens)]) / len(gt_tokens)
            return ce
    return None

In [56]:
# Compute cross-entropy for each result (answers only)
for i, res in enumerate(results):
    gt = res['ground_truth']
    pred = extract_answer_from_prediction(res['prediction'] or '')
    logprobs = res['logprobs']
    print(f"sample {i+1} : gt = {gt}, pred = {pred}")
    ce = compute_cross_entropy_from_logprobs(gt, pred, logprobs)
    print(f"Sample {i+1} Cross-Entropy (answer only): {ce if ce is not None else 'N/A'}")

sample 1 : gt = 0, pred = 5
Sample 1 Cross-Entropy (answer only): N/A
sample 2 : gt = 19, pred = a) $S$ is infinite.
b) The highest value of $f(k,p)$ for $k \geq 1$ and $p \in S$ is 27.
Sample 2 Cross-Entropy (answer only): N/A
sample 3 : gt = 256, pred = The car can travel 256 miles on 8 gallons of gas.
Sample 3 Cross-Entropy (answer only): N/A


In [ ]:
# --- Answer-match evaluator (replaces cross-entropy) ---
import re
import json

def _normalize_answer(s: str) -> str:
    if s is None:
        return ""
    # remove common LaTeX \text{...} wrappers
    s = re.sub(r"\\text\{([^}]*)\}", r"\1", s)
    # remove dollars and surrounding math markers
    s = s.replace("$", "")
    # collapse whitespace and lowercase
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _extract_number(s: str):
    if not s:
        return None
    s2 = re.sub(r"[^0-9eE+\-\.]+", " ", s)
    m = re.search(r"[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?", s2)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None


def answer_match_from_prediction(gt_answer: str, prediction_text: str) -> int:
    """
    Look for the model's answer between <answer>...</answer> in prediction_text.
    Return 1 if the extracted answer matches gt_answer (after light normalization),
    otherwise return 0.
    Matching strategy:
    - exact normalized string match
    - numeric equality when both contain a parseable number
    """
    if not prediction_text or not gt_answer:
        return 0

    m = re.search(r"<answer>(.*?)</answer>", prediction_text, flags=re.S | re.I)
    if not m:
        return 0
    pred = m.group(1).strip()
    if not pred:
        return 0

    norm_gt = _normalize_answer(gt_answer)
    norm_pred = _normalize_answer(pred)
    if norm_gt == norm_pred:
        return 1

    # try numeric comparison (robust to formatting differences)
    gnum = _extract_number(gt_answer)
    pnum = _extract_number(pred)
    if gnum is not None and pnum is not None:
        # consider approximate equality for floats
        if abs(gnum - pnum) <= 1e-8 * max(1.0, abs(gnum)):
            return 1
    return 0


def batch_evaluate_match(gt_answers, predictions):
    """
    Evaluate a list of ground-truth answers against a list of model prediction strings.
    - gt_answers: list of strings (ground truth)
    - predictions: list of strings (raw model outputs containing <answer> tags)
    Returns: list of 1/0 matches
    """
    results = []
    for gt, pred_text in zip(gt_answers, predictions):
        results.append(answer_match_from_prediction(gt, pred_text))
    return results

# Example usage: pass your ground-truth list and the raw prediction texts.
# prediction_texts = [res.get('output_text') or res.get('text') or json.dumps(res) for res in results]
# matches = batch_evaluate_match(gt_answers, prediction_texts)
# print(matches)
